In [1]:
# ============================================================
# EXERCISE 8: KOHONEN SELF-ORGANIZING MAP (SOM)
# Customer Segmentation and Cluster Exploration System
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA


# ============================================================
# 1. LOAD MALL CUSTOMERS DATASET
# ============================================================

df = pd.read_csv("Mall_Customers1.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst five rows:")
print(df.head())


# ============================================================
# 2. PREPROCESS DATA
# ============================================================

# Identify numerical attributes
numerical_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

# Use customer attributes, excluding ID columns
feature_columns = [
    col for col in numerical_columns
    if col.lower() not in ["customerid", "customer_id", "id"]
]

if len(feature_columns) < 2:
    raise ValueError(
        "The dataset must contain at least two numerical customer features."
    )

X = df[feature_columns].copy()

# Handle missing values
X = X.fillna(X.median())

# Normalize numerical attributes
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

n_samples, n_features = X_scaled.shape

print("\nNumerical features used:", feature_columns)
print("Number of customers:", n_samples)
print("Number of features:", n_features)


# ============================================================
# 3. SOM IMPLEMENTATION
# ============================================================

class SOM:

    def __init__(
        self,
        rows=5,
        cols=5,
        learning_rate=0.5,
        sigma=None,
        epochs=100
    ):

        self.rows = rows
        self.cols = cols
        self.learning_rate = learning_rate
        self.epochs = epochs

        self.sigma = (
            max(rows, cols) / 2
            if sigma is None else sigma
        )

        self.weights = None
        self.history = []

    def initialize(self, X):

        rng = np.random.default_rng(42)

        indices = rng.choice(
            len(X),
            size=self.rows * self.cols,
            replace=(self.rows * self.cols > len(X))
        )

        self.weights = X[indices].copy().reshape(
            self.rows,
            self.cols,
            X.shape[1]
        )

    def find_bmu(self, sample):

        distances = np.linalg.norm(
            self.weights - sample,
            axis=2
        )

        return np.unravel_index(
            np.argmin(distances),
            distances.shape
        )

    def fit(self, X):

        self.initialize(X)

        rng = np.random.default_rng(42)

        for epoch in range(self.epochs):

            # Gradually reduce learning rate and neighborhood
            progress = epoch / max(self.epochs - 1, 1)

            lr = self.learning_rate * (1 - progress)

            sigma_now = max(
                self.sigma * (1 - progress),
                0.5
            )

            for index in rng.permutation(len(X)):

                sample = X[index]

                bmu = self.find_bmu(sample)

                for i in range(self.rows):
                    for j in range(self.cols):

                        grid_distance = (
                            (i - bmu[0]) ** 2
                            + (j - bmu[1]) ** 2
                        )

                        # Neighborhood influence
                        influence = np.exp(
                            -grid_distance
                            / (2 * sigma_now ** 2)
                        )

                        # Competitive learning update
                        self.weights[i, j] += (
                            lr
                            * influence
                            * (sample - self.weights[i, j])
                        )

            self.history.append(
                self.quantization_error(X)
            )

        return self

    def predict(self, X):

        return np.array([
            self.find_bmu(sample)
            for sample in X
        ])

    def quantization_error(self, X):

        distances = []

        for sample in X:

            bmu = self.find_bmu(sample)

            distance = np.linalg.norm(
                sample - self.weights[bmu]
            )

            distances.append(distance)

        return float(np.mean(distances))


# ============================================================
# 4. TRAIN DEFAULT SOM
# ============================================================

som = SOM(
    rows=5,
    cols=5,
    learning_rate=0.5,
    epochs=100
)

som.fit(X_scaled)

bmus = som.predict(X_scaled)

print("\nSOM training completed!")
print("SOM grid:", som.rows, "x", som.cols)
print("Quantization error:", round(
    som.quantization_error(X_scaled), 4
))


# ============================================================
# 5. VISUALIZATION HELPERS
# ============================================================

def plot_som_clusters(model, X, labels, title):

    coords = model.predict(X)

    plt.figure(figsize=(8, 6))

    scatter = plt.scatter(
        coords[:, 1],
        coords[:, 0],
        c=labels,
        cmap="tab10",
        s=55,
        edgecolors="black"
    )

    plt.gca().invert_yaxis()

    plt.xticks(range(model.cols))
    plt.yticks(range(model.rows))

    plt.xlabel("SOM Grid Column")
    plt.ylabel("SOM Grid Row")
    plt.title(title)

    plt.colorbar(scatter, label="Cluster")

    plt.grid(alpha=0.25)
    plt.show()


def get_cluster_labels(model, X):

    coords = model.predict(X)

    return np.array([
        r * model.cols + c
        for r, c in coords
    ])


# ============================================================
# OPTION 1: CUSTOMER SEGMENTATION USING SOM
# ============================================================

def option1():

    labels = get_cluster_labels(som, X_scaled)

    result = df.copy()
    result["SOM_Row"] = bmus[:, 0]
    result["SOM_Column"] = bmus[:, 1]
    result["SOM_Cluster"] = labels

    print("\n--- CUSTOMER SEGMENTATION USING SOM ---")
    print(result.head(15).to_string(index=False))

    print("\nCustomers per occupied SOM neuron:")
    print(
        result.groupby(
            ["SOM_Row", "SOM_Column"]
        ).size().sort_values(ascending=False).head(15)
    )

    plot_som_clusters(
        som,
        X_scaled,
        labels,
        "Customer Segmentation using SOM"
    )


# ============================================================
# OPTION 2: BEST MATCHING UNIT EXPLORER
# ============================================================

def option2():

    index = int(input(
        f"\nEnter customer row index (0-{n_samples - 1}): "
    ))

    if not 0 <= index < n_samples:
        print("Invalid customer index.")
        return

    sample = X_scaled[index]
    bmu = som.find_bmu(sample)

    distance = np.linalg.norm(
        sample - som.weights[bmu]
    )

    print("\n--- BMU EXPLORER ---")
    print("Customer row:", index)
    print(df.iloc[index].to_dict())
    print("Best Matching Unit:", bmu)
    print("Distance to BMU:", round(distance, 4))


# ============================================================
# OPTION 3: DYNAMIC CLUSTER COUNT CHALLENGE
# ============================================================

def option3():

    print("\n--- DYNAMIC CLUSTER COUNT CHALLENGE ---")

    for k in [2, 3, 4, 5, 6]:

        model = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )

        labels = model.fit_predict(X_scaled)

        score = silhouette_score(
            X_scaled,
            labels
        )

        print(
            f"Clusters: {k} | "
            f"Silhouette Score: {score:.4f} | "
            f"Inertia: {model.inertia_:.2f}"
        )

    print(
        "\nNote: This option compares K-Means cluster counts "
        "as a reference for exploring segmentation choices."
    )


# ============================================================
# OPTION 4: CUSTOMER CLUSTER MIGRATION ANALYZER
# ============================================================

def option4():

    print("\n--- CUSTOMER CLUSTER MIGRATION ANALYZER ---")

    # Compare current 5x5 SOM with a 4x4 SOM
    alternative = SOM(
        rows=4,
        cols=4,
        learning_rate=0.5,
        epochs=100
    ).fit(X_scaled)

    current_coords = som.predict(X_scaled)
    alternative_coords = alternative.predict(X_scaled)

    current_labels = (
        current_coords[:, 0] * som.cols
        + current_coords[:, 1]
    )

    alternative_labels = (
        alternative_coords[:, 0] * alternative.cols
        + alternative_coords[:, 1]
    )

    moved = current_labels != alternative_labels

    print("Customers compared:", n_samples)
    print("Different neuron index:", int(moved.sum()))
    print(
        "Percentage with changed neuron index:",
        round(moved.mean() * 100, 2),
        "%"
    )

    print(
        "\nNote: Grid sizes differ, so this compares neuron assignments "
        "by flattened index, not identical physical locations."
    )


# ============================================================
# OPTION 5: CLUSTER PROTOTYPE GENERATOR
# ============================================================

def option5():

    print("\n--- CLUSTER PROTOTYPE GENERATOR ---")

    print("Feature order:", feature_columns)

    for i in range(som.rows):
        for j in range(som.cols):

            customers = np.where(
                (bmus[:, 0] == i)
                & (bmus[:, 1] == j)
            )[0]

            if len(customers) == 0:
                continue

            # Convert prototype back to original feature scale
            prototype = scaler.inverse_transform(
                som.weights[i, j].reshape(1, -1)
            )[0]

            print(
                f"\nNeuron ({i}, {j}) | "
                f"Customers: {len(customers)}"
            )

            print(
                dict(zip(
                    feature_columns,
                    np.round(prototype, 2)
                ))
            )


# ============================================================
# OPTION 6: OUTLIER CUSTOMER DETECTOR
# ============================================================

def option6():

    print("\n--- OUTLIER CUSTOMER DETECTOR ---")

    distances = np.array([
        np.linalg.norm(
            X_scaled[i] - som.weights[bmus[i]]
        )
        for i in range(n_samples)
    ])

    threshold = np.mean(distances) + 2 * np.std(distances)

    outliers = np.where(
        distances > threshold
    )[0]

    print("Outlier threshold:", round(threshold, 4))
    print("Outlier customers found:", len(outliers))

    if len(outliers) == 0:
        print("No outliers found using this threshold.")
        return

    result = df.iloc[outliers].copy()
    result["BMU_Distance"] = distances[outliers]

    print(result.head(20).to_string(index=False))


# ============================================================
# OPTION 7: SOM CLUSTER QUALITY COMPARISON
# ============================================================

def option7():

    print("\n--- SOM CLUSTER QUALITY COMPARISON ---")

    print(
        "5x5 SOM quantization error:",
        round(som.quantization_error(X_scaled), 4)
    )

    smaller_som = SOM(
        rows=3,
        cols=3,
        learning_rate=0.5,
        epochs=100
    ).fit(X_scaled)

    larger_som = SOM(
        rows=7,
        cols=7,
        learning_rate=0.5,
        epochs=100
    ).fit(X_scaled)

    print(
        "3x3 SOM quantization error:",
        round(smaller_som.quantization_error(X_scaled), 4)
    )

    print(
        "7x7 SOM quantization error:",
        round(larger_som.quantization_error(X_scaled), 4)
    )

    plt.figure(figsize=(8, 4))

    plt.plot(
        som.history,
        label="5x5 SOM"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Quantization Error")
    plt.title("SOM Training Progress")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


# ============================================================
# MAIN MENU
# ============================================================

while True:

    print("\n" + "=" * 60)
    print(" KOHONEN SOM CUSTOMER SEGMENTATION SYSTEM ")
    print("=" * 60)

    print("1. Customer Segmentation using SOM")
    print("2. Best Matching Unit Explorer")
    print("3. Dynamic Cluster Count Challenge")
    print("4. Customer Cluster Migration Analyzer")
    print("5. Cluster Prototype Generator")
    print("6. Outlier Customer Detector")
    print("7. SOM Cluster Quality Comparison")
    print("8. Exit")

    choice = input("\nEnter your choice: ")

    if choice == "1":
        option1()
    elif choice == "2":
        option2()
    elif choice == "3":
        option3()
    elif choice == "4":
        option4()
    elif choice == "5":
        option5()
    elif choice == "6":
        option6()
    elif choice == "7":
        option7()
    elif choice == "8":
        print("\nExiting SOM system...")
        break
    else:
        print("Invalid choice. Please enter 1-8.")

Dataset loaded successfully!
Dataset shape: (200, 5)

Columns: ['CustomerID', 'Gender', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']

First five rows:
   CustomerID  Gender  Age  Annual Income (k$)  Spending Score (1-100)
0           1    Male   19                  15                      39
1           2    Male   21                  15                      81
2           3  Female   20                  16                       6
3           4  Female   23                  16                      77
4           5  Female   31                  17                      40

Numerical features used: ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
Number of customers: 200
Number of features: 3

SOM training completed!
SOM grid: 5 x 5
Quantization error: 0.4584

 KOHONEN SOM CUSTOMER SEGMENTATION SYSTEM 
1. Customer Segmentation using SOM
2. Best Matching Unit Explorer
3. Dynamic Cluster Count Challenge
4. Customer Cluster Migration Analyzer
5. Cluster Prototype Generator
6


Enter your choice:  2

Enter customer row index (0-199):  4



--- BMU EXPLORER ---
Customer row: 4
{'CustomerID': 5, 'Gender': 'Female', 'Age': 31, 'Annual Income (k$)': 17, 'Spending Score (1-100)': 40}
Best Matching Unit: (np.int64(4), np.int64(1))
Distance to BMU: 0.6923

 KOHONEN SOM CUSTOMER SEGMENTATION SYSTEM 
1. Customer Segmentation using SOM
2. Best Matching Unit Explorer
3. Dynamic Cluster Count Challenge
4. Customer Cluster Migration Analyzer
5. Cluster Prototype Generator
6. Outlier Customer Detector
7. SOM Cluster Quality Comparison
8. Exit



Enter your choice:  3



--- DYNAMIC CLUSTER COUNT CHALLENGE ---


C:\Users\Venky  Tammisetty\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\Venky  Tammisetty\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\Venky  Tammisetty\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Clusters: 2 | Silhouette Score: 0.3355 | Inertia: 389.39
Clusters: 3 | Silhouette Score: 0.3578 | Inertia: 295.21
Clusters: 4 | Silhouette Score: 0.4040 | Inertia: 205.23
Clusters: 5 | Silhouette Score: 0.4166 | Inertia: 168.25
Clusters: 6 | Silhouette Score: 0.4284 | Inertia: 133.87

Note: This option compares K-Means cluster counts as a reference for exploring segmentation choices.

 KOHONEN SOM CUSTOMER SEGMENTATION SYSTEM 
1. Customer Segmentation using SOM
2. Best Matching Unit Explorer
3. Dynamic Cluster Count Challenge
4. Customer Cluster Migration Analyzer
5. Cluster Prototype Generator
6. Outlier Customer Detector
7. SOM Cluster Quality Comparison
8. Exit


C:\Users\Venky  Tammisetty\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(
C:\Users\Venky  Tammisetty\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(



Enter your choice:  8



Exiting SOM system...
